# ML Model Comparison – Workflow Delay Prediction
**Option C: 5-Model Comparison**  
Christ University | MCA 6th Trimester – Workflow Automation System

This notebook trains and compares 5 ML classifiers on synthetic service-request data:
1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. Support Vector Machine (SVM)
5. Neural Network (MLP)

---

In [ ]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
DATA_PATH    = '../data/raw/synthetic_requests.csv'

# Colour palette for 5 models
MODEL_COLOURS = {
    'Logistic Regression': '#3b82f6',
    'Random Forest':       '#10b981',
    'Gradient Boosting':   '#f59e0b',
    'SVM':                 '#8b5cf6',
    'Neural Network':      '#ef4444',
}

plt.rcParams.update({
    'figure.facecolor': '#1e293b',
    'axes.facecolor':   '#0f172a',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#cbd5e1',
    'text.color':       '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#1e3a5f',
    'legend.facecolor': '#1e293b',
    'legend.edgecolor': '#334155',
})

print('✅ Imports complete')

In [ ]:
# ── Cell 2: Load & Explore Data ───────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'\nColumns:\n{list(df.columns)}')
print(f'\nClass balance:\n{df["is_delayed"].value_counts()}')
print(f'\nDelay rate: {df["is_delayed"].mean()*100:.1f}%')

df.head()

In [ ]:
# ── Cell 3: Exploratory Data Analysis ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Exploratory Data Analysis', fontsize=14, y=1.01)

# 1. Delay rate by request type
ax = axes[0, 0]
delay_by_type = df.groupby('request_type')['is_delayed'].mean().sort_values(ascending=False)
colours = ['#ef4444' if v > 0.5 else '#f59e0b' if v > 0.3 else '#10b981' for v in delay_by_type]
delay_by_type.plot(kind='bar', ax=ax, color=colours, edgecolor='none')
ax.set_title('Delay Rate by Request Type')
ax.set_ylabel('Delay Rate')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis='x', rotation=30)

# 2. Distribution of total_duration_hours
ax = axes[0, 1]
df[df['is_delayed']==0]['total_duration_hours'].hist(
    ax=ax, bins=40, alpha=0.6, color='#10b981', label='On-time')
df[df['is_delayed']==1]['total_duration_hours'].hist(
    ax=ax, bins=40, alpha=0.6, color='#ef4444', label='Delayed')
ax.set_title('Duration Distribution by Outcome')
ax.set_xlabel('Hours')
ax.legend()

# 3. Delay by priority
ax = axes[0, 2]
priority_labels = {1: 'Low', 2: 'Medium', 3: 'High'}
dr = df.groupby('priority')['is_delayed'].mean()
dr.index = [priority_labels.get(i, i) for i in dr.index]
dr.plot(kind='bar', ax=ax, color=['#10b981', '#f59e0b', '#ef4444'], edgecolor='none')
ax.set_title('Delay Rate by Priority')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis='x', rotation=0)

# 4. Hourly submission pattern
ax = axes[1, 0]
hourly = df.groupby('created_hour')['is_delayed'].mean()
ax.plot(hourly.index, hourly.values, color='#3b82f6', linewidth=2, marker='o', markersize=4)
ax.fill_between(hourly.index, hourly.values, alpha=0.2, color='#3b82f6')
ax.set_title('Delay Rate by Hour of Day')
ax.set_xlabel('Hour')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.axhline(df['is_delayed'].mean(), color='#ef4444', linestyle='--', linewidth=1, label='Average')
ax.legend()

# 5. Handler workload vs delay
ax = axes[1, 1]
wl = df.groupby('handler_workload')['is_delayed'].mean().reset_index()
ax.scatter(wl['handler_workload'], wl['is_delayed'], color='#8b5cf6', s=60, alpha=0.8)
ax.set_title('Handler Workload vs Delay Rate')
ax.set_xlabel('Handler Workload')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# 6. SLA hours distribution
ax = axes[1, 2]
df['sla_hours'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#f59e0b', edgecolor='none')
ax.set_title('SLA Distribution')
ax.set_xlabel('SLA Hours')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../data/eda_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved to data/eda_analysis.png')

In [ ]:
# ── Cell 4: Feature Engineering & Train/Test Split ────────────────────────────
df_c = df[df['is_completed'] == 1].copy()
print(f'Training on {len(df_c)} completed requests')

le_type  = LabelEncoder()
le_stage = LabelEncoder()
df_c['request_type_encoded'] = le_type.fit_transform(df_c['request_type'])
df_c['final_stage_encoded']  = le_stage.fit_transform(df_c['final_stage'])

df_c['is_weekend']        = (df_c['created_day_of_week'] >= 5).astype(int)
df_c['is_peak_hour']      = df_c['created_hour'].isin([10, 11, 14, 15, 16]).astype(int)
df_c['is_business_hours'] = df_c['created_hour'].between(8, 17).astype(int)
df_c['is_high_priority']  = (df_c['priority'] == 3).astype(int)
df_c['is_low_priority']   = (df_c['priority'] == 1).astype(int)
df_c['high_workload']     = (df_c['handler_workload'] > 5).astype(int)
df_c['total_stage_time']  = (
    df_c['stage_created_duration'] + df_c['stage_assigned_duration'] +
    df_c['stage_verified_duration'] + df_c['stage_approved_duration'] +
    df_c['stage_processed_duration']
)

FEATURE_COLUMNS = [
    'request_type_encoded', 'priority', 'created_hour', 'created_day_of_week',
    'is_weekend', 'is_peak_hour', 'is_business_hours', 'is_high_priority',
    'is_low_priority', 'handler_workload', 'high_workload', 'sla_hours',
    'stage_created_duration', 'stage_assigned_duration', 'stage_verified_duration',
    'stage_approved_duration', 'stage_processed_duration', 'total_stage_time',
    'final_stage_encoded',
]

X = df_c[FEATURE_COLUMNS]
y = df_c['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'\nFeatures: {len(FEATURE_COLUMNS)}')
print(f'Train size: {len(X_train):,} | Test size: {len(X_test):,}')
print(f'Class balance (train) – Delayed: {y_train.sum()} ({y_train.mean()*100:.1f}%)')

In [ ]:
# ── Cell 5: Train All 5 Models ────────────────────────────────────────────────
MODELS = {
    'Logistic Regression': (LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), False),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, max_depth=10,
                                                    random_state=RANDOM_STATE, n_jobs=-1), False),
    'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                                        random_state=RANDOM_STATE), False),
    'SVM':                 (SVC(kernel='rbf', probability=True,
                                random_state=RANDOM_STATE), True),
    'Neural Network':      (MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                                          random_state=RANDOM_STATE), True),
}

results = {}

for name, (model, use_scaled) in MODELS.items():
    Xtr = X_train_s if use_scaled else X_train
    Xte = X_test_s  if use_scaled else X_test

    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:, 1]

    results[name] = {
        'model':       model,
        'use_scaled':  use_scaled,
        'accuracy':    accuracy_score(y_test, y_pred),
        'precision':   precision_score(y_test, y_pred),
        'recall':      recall_score(y_test, y_pred),
        'f1':          f1_score(y_test, y_pred),
        'roc_auc':     roc_auc_score(y_test, y_proba),
        'y_pred':      y_pred,
        'y_proba':     y_proba,
    }
    print(f'✅ {name:<22} Acc={results[name]["accuracy"]:.3f} '
          f'F1={results[name]["f1"]:.3f} AUC={results[name]["roc_auc"]:.3f}')

print('\n✅ All 5 models trained')

In [ ]:
# ── Cell 6: Comparison Table ──────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {
        'Accuracy':  f"{r['accuracy']*100:.2f}%",
        'Precision': f"{r['precision']*100:.2f}%",
        'Recall':    f"{r['recall']*100:.2f}%",
        'F1 Score':  f"{r['f1']*100:.2f}%",
        'ROC-AUC':   f"{r['roc_auc']*100:.2f}%",
    }
    for name, r in results.items()
}).T

best = max(results, key=lambda n: results[n]['f1'])
print(f'🏆 Best model (F1): {best} → {results[best]["f1"]*100:.2f}%\n')
metrics_df

In [ ]:
# ── Cell 7: Confusion Matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices – All 5 Models', fontsize=13)

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['On-time', 'Delayed'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8: ROC Curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, linewidth=2,
            label=f"{name} (AUC={r['roc_auc']:.3f})",
            color=MODEL_COLOURS[name])

ax.plot([0,1], [0,1], 'w--', linewidth=1, alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves – All 5 Models')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: Feature Importance + Metric Comparison Bar Chart ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A: Metric comparison grouped bar
ax = axes[0]
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metric_keys))
width = 0.15

for i, (name, r) in enumerate(results.items()):
    vals = [r[k] for k in metric_keys]
    ax.bar(x + i * width, vals, width=width,
           label=name, color=MODEL_COLOURS[name], alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'], fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# B: Feature importance of best tree model
ax = axes[1]
tree_models = [(n, r) for n, r in results.items()
               if hasattr(r['model'], 'feature_importances_')]

if tree_models:
    best_tree_name, best_tree_r = max(tree_models, key=lambda x: x[1]['f1'])
    importances = best_tree_r['model'].feature_importances_
    top_k = 10
    idx   = np.argsort(importances)[-top_k:][::-1]
    feat_names = [FEATURE_COLUMNS[i] for i in idx]
    feat_vals  = importances[idx]

    bars = ax.barh(range(top_k), feat_vals[::-1],
                   color=MODEL_COLOURS.get(best_tree_name, '#3b82f6'), alpha=0.85)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(feat_names[::-1], fontsize=9)
    ax.set_title(f'Top {top_k} Features ({best_tree_name})')
    ax.set_xlabel('Importance')
    ax.grid(axis='x', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Feature importance\nnot available',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Final Summary ─────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('  FINAL SUMMARY')
print('='*65)
print(f'\n  Dataset: {len(df_c):,} completed requests | {len(FEATURE_COLUMNS)} features')
print(f'  Train/Test split: 80/20 (stratified)\n')
for name, r in sorted(results.items(), key=lambda x: -x[1]['f1']):
    star = ' ⭐' if name == best else ''
    print(f'  {name:<22}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}{star}')
print(f'\n  Best model: {best} (saved as best_model.pkl)')
print('='*65)